# Web Scraping with BeautifulSoup

Extract structured text and tables from HTML while respecting website policies.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Parse HTML with BeautifulSoup.
- Select elements using CSS selectors and extract text/links.
- Convert repeated cards or rows into a DataFrame.
- Understand when scraping is inappropriate or unreliable.

This notebook parses local sample HTML. Live scraping is not required to learn the parsing steps.

In [1]:
from bs4 import BeautifulSoup
import pandas as pd

html = """
<html><body>
  <article class="place" data-id="a1"><h2>Central Library</h2><p class="district">Central</p><a href="/places/a1">Details</a></article>
  <article class="place" data-id="a2"><h2>Riverside Library</h2><p class="district">North</p><a href="/places/a2">Details</a></article>
</body></html>
"""
soup = BeautifulSoup(html, "html.parser")
print(soup.title)  # None: the sample HTML has no title element.
print([h.get_text(strip=True) for h in soup.select("article.place h2")])

None
['Central Library', 'Riverside Library']


## Parse repeated items
Prefer stable attributes and semantic elements over brittle positional selectors such as `div:nth-child(3)`. Real sites can change structure at any time.

In [2]:
records = []
for card in soup.select("article.place"):
    heading = card.select_one("h2")
    district = card.select_one(".district")
    link = card.select_one("a[href]")
    records.append({
        "source_id": card.get("data-id"),
        "name": heading.get_text(" ", strip=True) if heading else None,
        "district": district.get_text(" ", strip=True) if district else None,
        "detail_path": link.get("href") if link else None,
    })
places = pd.DataFrame(records)
display(places)

,source_id,name,district,detail_path
0,a1,Central Library,Central,/places/a1
1,a2,Riverside Library,North,/places/a2


## HTML tables and live requests
`pandas.read_html` can extract HTML tables, but it may require compatible parsing dependencies. For live collection, check the site's terms, robots policy, rate limits, privacy expectations, and applicable law first. Do not scrape login-protected or personal data without authorization.

In [3]:
table_html = """
<table><tr><th>Month</th><th>Count</th></tr>
<tr><td>Jan</td><td>120</td></tr><tr><td>Feb</td><td>132</td></tr></table>
"""
table_soup = BeautifulSoup(table_html, "html.parser")
rows = []
for tr in table_soup.select("tr")[1:]:
    cells = [td.get_text(strip=True) for td in tr.select("td")]
    rows.append(cells)
table_df = pd.DataFrame(rows, columns=["month", "count"])
table_df["count"] = pd.to_numeric(table_df["count"])
display(table_df)

# Live retrieval pattern only; keep disabled by default.
# response = requests.get("https://example.com", timeout=10)
# response.raise_for_status()
# live_soup = BeautifulSoup(response.text, "html.parser")

,month,count
0,Jan,120
1,Feb,132


## Practice
Extract each item name and numeric price from a small HTML snippet. Decide how your code should behave when an item has no price.

In [4]:
html_prices = """<ul><li class='item'><span class='name'>Notebook</span><span class='price'>4.5</span></li>
<li class='item'><span class='name'>Pen</span></li></ul>"""
soup_prices = BeautifulSoup(html_prices, "html.parser")
items = []
for item in soup_prices.select("li.item"):
    name = item.select_one(".name")
    price = item.select_one(".price")
    items.append({"name": name.get_text(strip=True) if name else None,
                  "price": float(price.get_text(strip=True)) if price else None})
display(pd.DataFrame(items))

,name,price
0,Notebook,4.5
1,Pen,NaN
